<a href="https://colab.research.google.com/github/Likelipop/01-NLP/blob/main/Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision torchaudio tqdm scikit-learn


In [2]:
!pip install datasets transformers

# Mini-project 1

Apply Transformers & its variants for a text classification problem.

## Problem statement
- Given a set of Transformers variants and a text classification dataset. Your task is to make a complete pipeline from input to output.
- Your pipeline **SHOULD** include the following required components. Each component **SHOULD** be organized into different `class` object.
    - Load & discover dataset
    - Preprocess data
    - Tokenize data
    - Create a DataLoader
    - Build or load model
    - Create a training workflow
    - Set up hyperparameters for training procedure
    - Train model
    - Evaluate model
    - Infer model


In [15]:
# Imports
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from torch.optim import AdamW
from transformers import AutoModel
from tqdm import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

In [4]:
# Update the datasets library
!pip install --upgrade datasets

In [5]:
import re
from typing import Tuple, Dict, Any
from datasets import load_dataset, DatasetDict, Dataset
from transformers import AutoTokenizer, PreTrainedTokenizer

class MyDataset:
    """
    A dataset handler class that loads, preprocesses, and tokenizes text data for sentiment analysis.

    Attributes:
        dataset_name (str): Name of the dataset to load (e.g., 'imdb').
        model_name (str): Pretrained transformer model name for tokenizer.
        max_length (int): Maximum sequence length for tokenization.
        tokenizer (PreTrainedTokenizer): Hugging Face tokenizer.
        dataset (DatasetDict): Loaded dataset object.
    """

    def __init__(self, dataset_name: str, model_name: str, max_length: int) -> None:
        """
        Initializes the MyDataset object with dataset and tokenizer configurations.

        Args:
            dataset_name (str): Name of the dataset to load.
            model_name (str): Hugging Face model name to load the tokenizer from.
            max_length (int): Max token length for input sequences.
        """
        self.dataset_name = dataset_name
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer: PreTrainedTokenizer = AutoTokenizer.from_pretrained(model_name)
        self.dataset: DatasetDict | None = None

    def _clean_text(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Cleans the input text by removing punctuation, digits, and keeping only alphabetic words.

        Args:
            example (Dict[str, Any]): A single data sample from the dataset.

        Returns:
            Dict[str, Any]: Cleaned data sample.
        """
        text = example["text"].lower()
        text = re.sub(r"[^\w\s]", "", text)  # Remove punctuation
        text = " ".join(word for word in text.split() if word.isalpha())  # Keep only alphabetic words
        example["text"] = text
        return example

    def _tokenize(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Tokenizes the cleaned text using the specified tokenizer.

        Args:
            example (Dict[str, Any]): A single data sample with cleaned text.

        Returns:
            Dict[str, Any]: Tokenized representation of the sample.
        """
        return self.tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

    def load(self) -> Tuple[Dataset, Dataset]:
        """
        Loads, cleans, tokenizes, and formats the dataset for training and testing.

        Returns:
            Tuple[Dataset, Dataset]: Tuple containing PyTorch-ready training and test datasets.
        """
        self.dataset = load_dataset(self.dataset_name)
        train_dataset = self.dataset["train"]
        test_dataset = self.dataset["test"]

        # Clean text
        train_dataset = train_dataset.map(self._clean_text)
        test_dataset = test_dataset.map(self._clean_text)

        # Tokenize
        train_dataset = train_dataset.map(self._tokenize, batched=True)
        test_dataset = test_dataset.map(self._tokenize, batched=True)

        # Set format for PyTorch
        train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
        test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

        return train_dataset, test_dataset


In [21]:
config = {
    "MODEL_NAME" : "distilbert-base-uncased",
    "MAX_LENGTH" : 256
}

dataset = MyDataset(
    dataset_name = "imdb",
    model_name = config["MODEL_NAME"],
    max_length = config["MAX_LENGTH"]
)

train_dataset, test_dataset = dataset.load()

In [7]:
import pprint

pprint.pprint(dataset.dataset["train"])
pprint.pprint(dataset.dataset["train"][0])

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})
{'label': 0,
 'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the '
         'controversy that surrounded it when it was first released in 1967. I '
         'also heard that at first it was seized by U.S. customs if it ever '
         'tried to enter this country, therefore being a fan of films '
         'considered "controversial" I really had to see this for myself.<br '
         '/><br />The plot is centered around a young Swedish drama student '
         'named Lena who wants to learn everything she can about life. In '
         'particular she wants to focus her attentions to making some sort of '
         'documentary on what the average Swede thought about certain '
         'political issues such as the Vietnam War and race issues in the '
         'United States. In between asking politicians and ordinary denizens '
         'of Stockholm about their opinions on politics, she has sex w

In [8]:
pprint.pprint(train_dataset)
pprint.pprint(train_dataset[1])

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 25000
})
{'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'input_ids': tensor([  10

In [9]:
import torch
from torch import nn

# 1. Get DataLoaders
import multiprocessing
num_workers = multiprocessing.cpu_count()

In [10]:
class DataLoaderManager:
    def __init__(self, train_dataset, test_dataset, batch_size=16, nw = 1):
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        self.batch_size = batch_size
        self.nw = nw

    def get_loaders(self):
        train_loader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.nw)
        test_loader = DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=4)
        return train_loader, test_loader



### BERT
- BERT is a deep learning language model designed to improve the efficiency of natural language processing (NLP) tasks. It is famous for its ability to consider context by analyzing the relationships between words in a sentence bidirectionally. It was introduced by Google researchers in a 2018 paper titled “BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.” Since then, the BERT model has been fine-tuned for use in a variety of fields, including biology, data science, and medicine.

- You could discover [BERT documentation](https://huggingface.co/docs/transformers/en/model_doc/bert) from transformers library @ Huggingface for more details.

- Examples of training use case for Huggingface model: [Huggingface training](https://huggingface.co/docs/transformers/en/training)

In [11]:
class BERTClassifier(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.3, freeze_bert: bool = True):
        super(BERTClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)

        # Freeze BERT nếu cần
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits


In [17]:
class Trainer:
    def __init__(self, model, train_loader, test_loader, lr=2e-5, epochs=3):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.optimizer = AdamW(self.model.parameters(), lr=lr)
        self.epochs = epochs
        self.criterion = nn.CrossEntropyLoss()
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.95)
        self.scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    def train(self):
        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            loop = tqdm(self.train_loader, desc=f"Epoch {epoch+1}")

            for batch in loop:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                self.optimizer.zero_grad()

                if self.scaler:
                    with torch.cuda.amp.autocast():
                        outputs = self.model(input_ids, attention_mask)
                        loss = self.criterion(outputs, labels)
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    outputs = self.model(input_ids, attention_mask)
                    loss = self.criterion(outputs, labels)
                    loss.backward()
                    self.optimizer.step()

                total_loss += loss.item()
                loop.set_postfix(loss=loss.item())

            avg_loss = total_loss / len(self.train_loader)
            print(f"Epoch {epoch+1} finished. Avg loss: {avg_loss:.4f}")
            torch.cuda.empty_cache()
            self.scheduler.step()

    def evaluate(self):
        self.model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in self.test_loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(input_ids, attention_mask)
                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        print(f"Test Accuracy: {acc:.4f}")
        return acc


In [13]:
train_dataset = train_dataset[: int(len(train_dataset)/10)]

In [ ]:
# Use .select() to create a subset of the training dataset
# This is a more robust way to handle slicing with the datasets library
train_dataset_subset = train_dataset.select(range(0, int(len(train_dataset)/10)))

# Initialize the DataLoaderManager with the subset of the training dataset
# and the full test dataset.
# Ensure num_workers is appropriate for your system and dataset size.
loader = DataLoaderManager(train_dataset_subset, test_dataset, batch_size=16, nw = 2)
train_loader, test_loader = loader.get_loaders()

# 2. Build model
# freeze_bert=True: chỉ train classifier
model = BERTClassifier("bert-base-uncased", freeze_bert=True)


# 3. Train and evaluate
trainer = Trainer(model=model, train_loader=train_loader, test_loader=test_loader,
                  lr=2e-5, epochs=3)

trainer.train()
trainer.evaluate()

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 1:  65%|██████▍   | 102/157 [30:56<16:39, 18.18s/it, loss=0.32]